In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [ ]:
train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
train_data.head()

In [ ]:
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
test_data.head()

In [ ]:
women = train_data.loc[train_data.Sex =='female']["Survived"]
rate_women = sum(women)/len(women)
print("% of women who survived :", rate_women*100)

In [ ]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men =sum(men)/len(men)
print("% of men who survived :", rate_men*100)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]
X = train_data[features]
X_test = test_data[features]

In [ ]:
numeric_features = ['SibSp', 'Parch']
categorical_features = ['Pclass', 'Sex']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

In [ ]:
model = RandomForestClassifier(n_estimators = 100,max_depth = 5, random_state = 1)

pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', model)])

param_grid = {
    'classifier__n_estimators': [50, 100, 200],
    'classifier__max_depth': [None, 5, 10, 20],
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy')
grid_search.fit(X, y)

print("Best Parameters:", grid_search.best_params_)

In [ ]:
best_model = grid_search.best_estimator_
predictions = best_model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score


train_predictions = best_model.predict(X)
train_accuracy = accuracy_score(y, train_predictions)
print("Training Accuracy:", train_accuracy*100)

In [ ]:
output =pd.DataFrame({'PassengerId' : test_data.PassengerId, 'Survived' : predictions})
output.to_csv('submission.csv', index = False)